# Example 1.3.2 / 1: Defined Variables

Companion notebook for **1.3.2 Defined Variables and Initial Values** in the AMPL lab Documentation, section "Defined Variables". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Defined Variables

A defined variable names an expression in other variables. It makes long objectives and
constraints readable, and its value can be read after solving.

In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`. After
`ampl.solve(solver="highs")`, `ampl.var["Hours_Used"].value()` reads the defined variable like
any other, and `astatus()` gives its AMPL status: `sub`, because AMPL substituted it out of the
problem:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param rate {PROD} > 0;
param profit {PROD};
param market {PROD} >= 0;
param avail >= 0;

var Make {p in PROD} >= 0, <= market[p];
var Hours_Used = sum {p in PROD} Make[p] / rate[p];       # defined variable

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];
subject to Time: Hours_Used <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
hours = ampl.var["Hours_Used"]
print(hours.value(), hours.astatus())

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solved
40.0 sub


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; the `astatus` suffix gives the same status:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    30     4000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Hours_Used, Hours_Used.astatus;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
Hours_Used = 40
Hours_Used.astatus = sub

